In [5]:
import torch
from torch.utils.data import DataLoader
import numpy as np
import pickle
import pandas as pd

import networkx as nx

import matplotlib.pyplot as plt

from matplotlib.colors import PowerNorm
from matplotlib.colors import LogNorm
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap

from matrix_processing_helpers import sparsify_global_percentile
from plot_helpers import plot_rollout_graph, plot_feature_ranking_comparison

In [6]:
with open("data_processed/dataframes/input_cols_pruned.pkl", "rb") as f:
    INPUT_COLS_PRUNED = pickle.load(f)

feature_names = list(INPUT_COLS_PRUNED)

### SHAP

In [7]:
shap_vals = np.load("Results_MLP/shap_values/shap_values.npz")
shap_importance = np.mean(np.abs(shap_vals["shap_vals"]), axis=0)

In [10]:
df_shap = pd.DataFrame({
    'feature': feature_names,
    'score': shap_importance,
    'original_position': range(len(feature_names))
})

df_shap.to_csv("Results_MLP/shap_values/tables/shap_scores_unsorted.csv", 
          index=False, float_format="%.6f")

# Sort by value descending
df_sorted_shap = df_shap.sort_values(by='score', ascending=False).reset_index(drop=True)
df_sorted_shap['ranking'] = np.arange(1, len(feature_names)+1)

df_sorted_shap.to_csv("Results_MLP/shap_values/tables/shap_scores.csv", 
          index=False, float_format="%.6f")

In [9]:
df_sorted_shap

,feature,score,original_position,ranking
0,Albumin (g/dl),0.135676,9,1
1,3-Hydroxyisobutyric acid (mM),0.107704,4,2
2,"1,5-Anhydrosorbitol (mM)",0.091918,0,3
3,LDL cholesterol (mg/dl),0.088415,43,4
4,Glucose (mM),0.086184,32,5
...,...,...,...,...
67,Arginine (mM),0.007416,11,68
68,Trimethylamine-N-oxide (mM),0.007194,67,69
69,Methanol (mM),0.006091,50,70
70,C reactive protein (mg/dl),0.006063,17,71


In [ ]:
NUM_LAYERS = [3, 5, 7, 9, 11, 13, 15, 17, 20] 

for layer in NUM_LAYERS:
    shap_vals = np.load(f"Results/shap_values/values/{layer}layers/shap_values_{layer}layers.npz")
    shap_importance = np.mean(np.abs(shap_vals["shap_vals"]), axis=0)

    df_shap = pd.DataFrame({'feature': feature_names,
                            'score': shap_importance,
                            'original_position': range(len(feature_names))
                           })

    df_sorted_shap = df_shap.sort_values(by='score', ascending=False).reset_index(drop=True)
    df_sorted_shap['ranking'] = np.arange(1, len(feature_names)+1)

    df_shap.to_csv(f"Results/shap_values/tables/{layer}layers/shap_scores{layer}layers_unsorted.csv", 
          index=False, float_format="%.6f")

    df_sorted_shap.to_csv(f"Results/shap_values/tables/{layer}layers/shap_scores{layer}layers.csv", 
          index=False, float_format="%.6f")